# 62 · Async LangGraph: Concurrent Tool Calls with asyncio

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/esturban/agent/blob/main/examples/62-async-langgraph/async_langgraph_workbook.ipynb)

## What you'll learn
- Write `async def` LangGraph nodes
- Call `ainvoke()` and `astream_events()` instead of the sync equivalents
- Use `asyncio.gather()` inside a node to run tools concurrently
- See exactly *when* async saves wall-clock time vs when it doesn't

**Prerequisite:** [01 · Basic Agent](../1-basic-local-rag/) | [28 · Parallel Subgraph](../28-parallel-subgraph/)

In [ ]:
# ── Environment Detection ───────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in {'Google Colab' if IN_COLAB else 'local environment'}")

In [ ]:
if IN_COLAB:
    !pip install -q langchain langchain-openai langgraph python-dotenv
    print("Installation complete.")
else:
    print("Local: deps already installed via requirements.txt")

In [ ]:
import os

if IN_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in Colab Secrets or .env"
print("API key loaded.")

In [ ]:
import asyncio
import time
from typing import TypedDict

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)
print("Imports ready.")

## Part 1 — Sync Baseline

A standard sync LangGraph graph. Each node runs `llm.invoke()`, which **blocks the thread** while waiting for the response.
We'll time one invocation here, then compare when we switch to async.

In [ ]:
class SyncState(TypedDict):
    question: str
    answer: str

def sync_agent(state: SyncState) -> dict:
    response = llm.invoke([HumanMessage(content=state["question"])])
    return {"answer": response.content}

sync_builder = StateGraph(SyncState)
sync_builder.add_node("agent", sync_agent)
sync_builder.add_edge(START, "agent")
sync_builder.add_edge("agent", END)
sync_app = sync_builder.compile()

t0 = time.perf_counter()
result = sync_app.invoke({"question": "Name one planet in one word.", "answer": ""})
elapsed = time.perf_counter() - t0
print(f"Sync invoke:  {elapsed:.2f}s  →  '{result['answer']}'")

## Part 2 — Async Nodes + `ainvoke()`

Switching to `async def` and `await llm.ainvoke()` lets the event loop yield while waiting for the API.  
The total latency for a **single call** is about the same — the benefit appears when many calls run concurrently (Part 3).

In [ ]:
class AsyncState(TypedDict):
    question: str
    answer: str

async def async_agent(state: AsyncState) -> dict:
    response = await llm.ainvoke([HumanMessage(content=state["question"])])
    return {"answer": response.content}

async_builder = StateGraph(AsyncState)
async_builder.add_node("agent", async_agent)
async_builder.add_edge(START, "agent")
async_builder.add_edge("agent", END)
async_app = async_builder.compile()

t0 = time.perf_counter()
result = await async_app.ainvoke({"question": "Name one planet in one word.", "answer": ""})
elapsed = time.perf_counter() - t0
print(f"Async ainvoke: {elapsed:.2f}s  →  '{result['answer']}'")
print("\nSingle-call latency is similar — the gain shows up in Part 3.")

## Part 3 — `asyncio.gather()` for Concurrent Tool Calls

Inside one node, we fire **three I/O-bound tools simultaneously** using `asyncio.gather()`.  
We simulate each tool as a 1-second async operation.  
- **Sequential approach**: 3 × 1 s = ~3 s  
- **Concurrent approach** (`gather`): ~1 s  

This is the primary reason to write async nodes.

In [ ]:
# Simulated I/O-bound tools (each takes ~1 s)
async def fetch_weather() -> str:
    await asyncio.sleep(1)
    return "Sunny, 22 °C"

async def fetch_news() -> str:
    await asyncio.sleep(1)
    return "Markets up 0.3 %"

async def fetch_calendar() -> str:
    await asyncio.sleep(1)
    return "3 meetings today"

# ── Sequential (await one at a time) ────────────────────────────────
async def sequential_node(state: dict) -> dict:
    weather  = await fetch_weather()
    news     = await fetch_news()
    calendar = await fetch_calendar()
    return {"context": f"{weather} | {news} | {calendar}"}

# ── Concurrent (all at once) ─────────────────────────────────────────
async def concurrent_node(state: dict) -> dict:
    weather, news, calendar = await asyncio.gather(
        fetch_weather(), fetch_news(), fetch_calendar()
    )
    return {"context": f"{weather} | {news} | {calendar}"}

# Time both
t0 = time.perf_counter()
await sequential_node({})
seq_time = time.perf_counter() - t0

t0 = time.perf_counter()
await concurrent_node({})
con_time = time.perf_counter() - t0

print(f"Sequential : {seq_time:.2f}s  (3 tools, one at a time)")
print(f"Concurrent : {con_time:.2f}s  (3 tools, asyncio.gather)")
print(f"Speedup    : {seq_time / con_time:.1f}×")

In [ ]:
class BriefingState(TypedDict):
    context: str
    summary: str

async def gather_context(state: BriefingState) -> dict:
    weather, news, calendar = await asyncio.gather(
        fetch_weather(), fetch_news(), fetch_calendar()
    )
    return {"context": f"Weather: {weather}. News: {news}. Calendar: {calendar}."}

async def summarise(state: BriefingState) -> dict:
    prompt = f"Give a one-sentence morning briefing from: {state['context']}"
    response = await llm.ainvoke([HumanMessage(content=prompt)])
    return {"summary": response.content}

briefing_builder = StateGraph(BriefingState)
briefing_builder.add_node("gather", gather_context)
briefing_builder.add_node("summarise", summarise)
briefing_builder.add_edge(START, "gather")
briefing_builder.add_edge("gather", "summarise")
briefing_builder.add_edge("summarise", END)
briefing_app = briefing_builder.compile()

t0 = time.perf_counter()
out = await briefing_app.ainvoke({"context": "", "summary": ""})
print(f"Full graph ({time.perf_counter() - t0:.2f}s): {out['summary']}")

## Part 4 — Streaming with `astream_events()`

`astream_events()` emits token-level events as the model generates them.  
Filter on `on_chat_model_stream` to print tokens as they arrive.

In [ ]:
class StreamState(TypedDict):
    question: str
    answer: str

async def stream_agent(state: StreamState) -> dict:
    response = await llm.ainvoke([HumanMessage(content=state["question"])])
    return {"answer": response.content}

stream_builder = StateGraph(StreamState)
stream_builder.add_node("agent", stream_agent)
stream_builder.add_edge(START, "agent")
stream_builder.add_edge("agent", END)
stream_app = stream_builder.compile()

print("Streaming tokens: ", end="", flush=True)
async for event in stream_app.astream_events(
    {"question": "List 3 programming languages in one sentence.", "answer": ""},
    version="v2",
):
    if event["event"] == "on_chat_model_stream":
        chunk = event["data"]["chunk"].content
        if chunk:
            print(chunk, end="", flush=True)
print("\n[stream complete]")

## Summary

| Pattern | API | When to use |
|---------|-----|-------------|
| Sync node + `invoke()` | `llm.invoke()` | CPU-bound, simple scripts |
| Async node + `ainvoke()` | `await llm.ainvoke()` | I/O-bound, safe for event loops |
| `asyncio.gather()` inside node | `await asyncio.gather(...)` | Multiple I/O calls per node |
| `astream_events()` | `async for event in ...` | Token streaming UIs |

**Rule of thumb:** If a node makes more than one external call, `asyncio.gather()` is almost always worth it.

## Exercise 1 — Convert a Sync Graph

The cell below defines a sync graph that calls the LLM twice sequentially.  
Rewrite it so both calls run concurrently with `asyncio.gather()` inside an async node.

In [ ]:
# Starter — sync, sequential LLM calls
class TranslateState(TypedDict):
    text: str
    spanish: str
    french: str

def translate_node(state: TranslateState) -> dict:
    es = llm.invoke([HumanMessage(content=f"Translate to Spanish: {state['text']}")])
    fr = llm.invoke([HumanMessage(content=f"Translate to French: {state['text']}")])
    return {"spanish": es.content, "french": fr.content}

# TODO: rewrite translate_node as async using asyncio.gather()
# TODO: build the graph with StateGraph + ainvoke()

## Exercise 2 — Add a Fourth Tool

Add a `fetch_stock()` async tool (1-second delay) to the `gather_context` node from Part 3.  
Verify the total time stays around 1 second, not 4.

In [ ]:
# Starter — extend the concurrent_node from Part 3
async def fetch_stock() -> str:
    await asyncio.sleep(1)
    return "AAPL +1.2 %"

# TODO: add fetch_stock() to asyncio.gather() in gather_context
# TODO: re-run the briefing graph and confirm time stays ~1 s

## What's Next

- **[28 · Parallel Subgraph](../28-parallel-subgraph/)** — fan-out at the graph level instead of inside a node
- **[72 · Batch Agent Runner](../72-batch-agent-runner/)** — pure-asyncio batch processing with retry
- **[LangGraph async docs](https://langchain-ai.github.io/langgraph/how-tos/async/)** — official async patterns

---
## Answer Key

In [ ]:
# Answer 1 — concurrent translate node
async def async_translate_node(state: TranslateState) -> dict:
    es_msg = HumanMessage(content=f"Translate to Spanish: {state['text']}")
    fr_msg = HumanMessage(content=f"Translate to French: {state['text']}")
    es, fr = await asyncio.gather(
        llm.ainvoke([es_msg]),
        llm.ainvoke([fr_msg]),
    )
    return {"spanish": es.content, "french": fr.content}

translate_builder = StateGraph(TranslateState)
translate_builder.add_node("translate", async_translate_node)
translate_builder.add_edge(START, "translate")
translate_builder.add_edge("translate", END)
translate_app = translate_builder.compile()

t0 = time.perf_counter()
out = await translate_app.ainvoke({"text": "Good morning", "spanish": "", "french": ""})
print(f"({time.perf_counter()-t0:.2f}s) ES: {out['spanish']}  |  FR: {out['french']}")

In [ ]:
# Answer 2 — four concurrent tools, still ~1 s
async def gather_context_4(state: BriefingState) -> dict:
    weather, news, calendar, stock = await asyncio.gather(
        fetch_weather(), fetch_news(), fetch_calendar(), fetch_stock()
    )
    return {"context": f"Weather: {weather}. News: {news}. Calendar: {calendar}. Stock: {stock}."}

b4_builder = StateGraph(BriefingState)
b4_builder.add_node("gather", gather_context_4)
b4_builder.add_node("summarise", summarise)
b4_builder.add_edge(START, "gather")
b4_builder.add_edge("gather", "summarise")
b4_builder.add_edge("summarise", END)
b4_app = b4_builder.compile()

t0 = time.perf_counter()
out = await b4_app.ainvoke({"context": "", "summary": ""})
print(f"4-tool gather ({time.perf_counter()-t0:.2f}s): {out['summary']}")